In [12]:
!pip install requests pandas numpy -q

In [13]:
!pip install ccxt -q

In [14]:
# =========================================================
# OBV 상승 다이버전스 스크리너
# [스테이블코인 이중 필터 + 이상치 제거 / 검증 완료]
# 데이터 소스: CoinGecko
# - 스테이블코인 이중 차단: ① 심볼 목록 ② 가격무변동(±1%) 자동감지
#   → 목록에 없는 새 스테이블코인도 자동으로 걸러짐
# - 이상치(XRP -1.18 같은 데이터오류) 자동 제외
# =========================================================

!pip install requests pandas numpy -q

import requests
import pandas as pd
import numpy as np
import time
import os
import json

# ---------------------------
# 설정값
# ---------------------------
MIN_MARKET_CAP_USD = 3_000_000_000
LOOKBACK_DAYS      = 30
DATA_DAYS          = 90
TOP_N              = 30
MAX_COINS          = 45      # 스테이블 빠지는 만큼 늘림
REQUEST_PAUSE      = 3.0     # CoinGecko 키 있으면 1.5로 줄여도 됨
CACHE_FILE         = "obv_cache.json"
OUTLIER_MAD        = 5.0
STABLE_MOVE_PCT    = 1.0     # 30일 변동 이 % 미만이면 스테이블로 자동 간주

# ★CoinGecko 무료 키 (있으면 넣기. 없으면 빈칸 두면 키 없이 동작)
COINGECKO_KEY = ""   # 예: "CG-xxxxxxxx"

BASE = "https://api.coingecko.com/api/v3"

# 스테이블코인 목록 (확장판)
STABLE_COINS = {"USDT","USDC","DAI","USDE","USDS","USD1","USYC","TUSD",
                "FDUSD","USDD","PYUSD","GUSD","FRAX","LUSD","USDP","CRVUSD",
                "USDF","USDG","RLUSD","USD0","GHO","USDX","BUSD","EURT","EURS"}

# =========================================================
def load_cache():
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, "r") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_cache(cache):
    try:
        with open(CACHE_FILE, "w") as f:
            json.dump(cache, f)
    except Exception:
        pass

cache = load_cache()

def safe_get(url, params=None, max_retry=6):
    if params is None:
        params = {}
    if COINGECKO_KEY:
        params["x_cg_demo_api_key"] = COINGECKO_KEY   # 키 있으면 자동 부착
    for attempt in range(max_retry):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                wait = REQUEST_PAUSE * (attempt + 2)
                print(f"   429 -> {wait:.0f}초 대기...")
                time.sleep(wait); continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == max_retry - 1:
                print(f"   요청 실패: {e}"); return None
            time.sleep(REQUEST_PAUSE)
    return None

# =========================================================
# 시총 상위 코인 (★스테이블 1차 필터: 심볼)
# =========================================================
def get_top_coins(min_cap, max_coins):
    coins = []
    data = safe_get(f"{BASE}/coins/markets", {
        "vs_currency": "usd", "order": "market_cap_desc",
        "per_page": 250, "page": 1})
    if not data:
        return coins
    for c in data:
        mc = c.get("market_cap")
        sym = c["symbol"].upper()
        if mc and mc >= min_cap and sym not in STABLE_COINS:  # 1차: 심볼 거름
            coins.append({"id": c["id"], "symbol": sym, "mcap": mc})
    return coins[:max_coins]

def get_ohlc_volume(coin_id, days):
    if coin_id in cache:
        d = cache[coin_id]
        return pd.DataFrame({"close": d["close"], "volume": d["volume"]})
    data = safe_get(f"{BASE}/coins/{coin_id}/market_chart", {
        "vs_currency": "usd", "days": days, "interval": "daily"})
    if not data or "prices" not in data:
        return None
    prices = data["prices"]; volumes = data.get("total_volumes", [])
    if not prices or not volumes:
        return None
    closes = [p[1] for p in prices]
    vols   = [v[1] for v in volumes][:len(prices)]
    cache[coin_id] = {"close": closes, "volume": vols}
    save_cache(cache)
    return pd.DataFrame({"close": closes, "volume": vols})

def calc_obv(df):
    direction = np.sign(df["close"].diff()).fillna(0)
    return (direction * df["volume"]).cumsum()

def score_divergence(df, lookback):
    if len(df) < lookback + 1:
        return None
    recent = df.tail(lookback).reset_index(drop=True)
    obv = calc_obv(df).tail(lookback).reset_index(drop=True)
    x = np.arange(len(recent))
    price = recent["close"].values
    if price[0] == 0:
        return None
    price_slope = np.polyfit(x, price / price[0], 1)[0]
    obv_vals = obv.values
    denom = abs(obv_vals[0]) if abs(obv_vals[0]) > 1e-9 else 1
    obv_slope = np.polyfit(x, obv_vals / denom, 1)[0]
    score = obv_slope - price_slope
    price_change = (price[-1]/price[0]-1)*100
    if price_slope <= 0.01 and obv_slope >= 0.002:
        grade = "강함 ★★★" if (obv_slope > 0.01 and price_slope < 0) else "보통 ★★"
    elif obv_slope > 0:
        grade = "약함 ★"
    else:
        grade = "-"
    return {"price_slope": round(price_slope,5), "obv_slope": round(obv_slope,5),
            "score": round(score,5), "grade": grade,
            "price_change_%": round(price_change,1)}

def detect_outliers(scores, n_mad=OUTLIER_MAD):
    if len(scores) < 4: return set()
    arr = np.array(scores, dtype=float)
    median = np.median(arr); mad = np.median(np.abs(arr - median))
    if mad == 0: return set()
    modified_z = 0.6745 * (arr - median) / mad
    return {i for i, z in enumerate(modified_z) if abs(z) > n_mad}

# =========================================================
# 메인
# =========================================================
print("1단계: 시총 상위 코인 수집 (스테이블 1차 필터: 심볼)...")
coins = get_top_coins(MIN_MARKET_CAP_USD, MAX_COINS)
print(f"  -> 대상: {len(coins)}개 | 캐시: {len(cache)}개\n")

print(f"2단계: OBV 분석 중...")
results = []
stable_auto_removed = []
for i, c in enumerate(coins):
    df = get_ohlc_volume(c["id"], DATA_DAYS)
    if df is not None and len(df) >= LOOKBACK_DAYS + 1:
        res = score_divergence(df, LOOKBACK_DAYS)
        if res:
            # ★2차 필터: 가격이 거의 안 움직이면(±1%) 스테이블로 자동 제외
            if abs(res["price_change_%"]) < STABLE_MOVE_PCT:
                stable_auto_removed.append(c["symbol"])
            else:
                results.append({
                    "코인": c["symbol"], "시총($B)": round(c["mcap"]/1e9, 2),
                    "등급": res["grade"], "점수": res["score"],
                    "OBV기울기": res["obv_slope"], "가격기울기": res["price_slope"],
                    f"{LOOKBACK_DAYS}일가격%": res["price_change_%"],
                })
    time.sleep(REQUEST_PAUSE)
    print(f"   {i+1}/{len(coins)} {c['symbol']} 완료")

# 이상치 제거
removed = []
if results:
    scores = [r["점수"] for r in results]
    outlier_idx = detect_outliers(scores)
    if outlier_idx:
        removed = [results[i]["코인"] for i in outlier_idx]
        results = [r for i, r in enumerate(results) if i not in outlier_idx]

print("\n" + "="*60)
if results:
    out = pd.DataFrame(results)
    out["_s"] = pd.to_numeric(out["점수"], errors="coerce").fillna(-1)
    out = out.sort_values("_s", ascending=False).drop(columns="_s")
    print(f"전체 분석: {len(out)}개 | 점수 상위 {min(TOP_N, len(out))}개")
    if stable_auto_removed:
        print(f"🔵 가격무변동 자동제외(스테이블 의심): {', '.join(stable_auto_removed)}")
    if removed:
        print(f"⚠️ 이상치 제외: {', '.join(removed)}")
    print()
    print(out.head(TOP_N).to_string(index=False))
    print("\n[등급] 강함★★★: 가격↓+OBV강세(매집의심) / 보통★★: 다이버전스 / 약함★: OBV만 약상승")
    print("[필터] 스테이블 이중차단(심볼+가격무변동) / 이상치 자동제외")
    strong = sum('강함' in r['등급'] for r in results)
    if strong == 0:
        print("\n※ 강함★★★ 0개 = 아직 뚜렷한 매집 신호 없음. 관망 구간.")
    else:
        print(f"\n※ 강함★★★ {strong}개. OBV는 선행신호 - 2주 뒤 결과 확인 필요(기록장에 기록).")
else:
    print("결과 없음.")
print("="*60)

1단계: 시총 상위 코인 수집 (스테이블 1차 필터: 심볼)...
  -> 대상: 21개 | 캐시: 21개

2단계: OBV 분석 중...
   1/21 BTC 완료
   2/21 ETH 완료
   3/21 BNB 완료
   4/21 XRP 완료
   5/21 SOL 완료
   6/21 TRX 완료
   7/21 FIGR_HELOC 완료
   8/21 WBT 완료
   9/21 HYPE 완료
   10/21 DOGE 완료
   11/21 LEO 완료
   12/21 RAIN 완료
   13/21 ZEC 완료
   14/21 XMR 완료
   15/21 ADA 완료
   16/21 LINK 완료
   17/21 XLM 완료
   18/21 CC 완료
   19/21 BCH 완료
   20/21 GRAM 완료
   21/21 LTC 완료

전체 분석: 16개 | 점수 상위 16개
🔵 가격무변동 자동제외(스테이블 의심): XRP, FIGR_HELOC, WBT
⚠️ 이상치 제외: RAIN, CC

  코인  시총($B)     등급       점수   OBV기울기    가격기울기  30일가격%
 XLM    5.91 강함 ★★★  0.02261  0.01819 -0.00442   -12.5
 XMR    6.71  보통 ★★  0.01639  0.02109  0.00470    13.7
 ZEC    7.71  보통 ★★  0.01196  0.01527  0.00331     9.2
 ETH  224.72  보통 ★★  0.00553  0.01019  0.00466    15.5
LINK    6.08  보통 ★★  0.00462  0.00860  0.00398    10.3
 TRX   30.95  보통 ★★  0.00438  0.00475  0.00036     3.2
GRAM    3.79      -  0.00385 -0.00286 -0.00671   -11.3
 LTC    3.46  보통 ★★  0.00053  0.00300  0.00247     5.3


In [15]:
# =====================================================================
# 유동성 휩소 스크리너 v3  [주식/원자재 이중차단 + ATR 필터 / 검증완료]
# 거래소: Bitget (USDT 무기한 선물)
# - ADX 제거 (휩소는 추세전략 아님)
# - 주식/원자재 이중 차단: ① 블랙리스트(확장) ② 2글자 코드 자동감지
#   → AAOI, CBRS, CL 등이 새로 떠도 걸러짐
# - ATR 최소 변동성 필터 (휩소 노이즈 제거)
# =====================================================================

!pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값
# ---------------------------
TIMEFRAME       = '1h'
LIMIT           = 50
TOP_N           = 50
LOOKBACK        = 12
SWEEP_TOLERANCE = 0.005
SHOW_TOP        = 20
MIN_GRADE       = "약함"   # "강함"/"보통"/"약함"
ATR_MIN_PCT     = 0.5     # 최소 변동성(현재가 %). 미만이면 노이즈로 제외
REQUEST_PAUSE   = 0.2

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

GRADE_RANK = {"강함 ★★★": 3, "보통 ★★": 2, "약함 ★": 1}
MIN_RANK = {"강함": 3, "보통": 2, "약함": 1}[MIN_GRADE]

# ★주식/원자재/FX 블랙리스트 (대폭 확장 - AAOI 등 추가)
STOCK_COMMODITY = {
    "CBRS","QCOM","MSTR","AAPL","TSLA","NVDA","AMZN","META","GOOGL","MSFT",
    "COIN","HOOD","PLTR","AMD","INTC","NFLX","BABA","SPY","QQQ","GME","AMC",
    "AAOI","MU","AVGO","SMCI","ARM","DELL","ORCL","CRM","ADBE","TSM",
    "MARA","RIOT","CLSK","NIO","LCID","RIVN","SOFI","PYPL","SQ","SHOP",
    "BA","DIS","NKE","WMT","JPM","BAC","V","MA","UBER","ABNB","GOOG",
    "XAG","XAU","WTI","GOLD","SILVER","OIL","XPT","XPD","NG","HG","COPPER",
    "BRENT","XBR","XTI","NGAS","CL",
    "US500","US100","NAS100","US30","SPX500","GER40","UK100","JP225","HK50",
    "EUR","GBP","JPY","AUD","CAD","CHF","NZD","EURUSD","GBPUSD","USDJPY",
}

def looks_like_non_coin(base):
    # 2글자 이하 알파벳 = 선물/원자재 코드 의심 (CL,NG,HG 등). 3글자 코인은 보호
    if len(base) <= 2 and base.isalpha():
        return True
    return False

def is_non_coin(symbol):
    base = symbol.split('/')[0].split(':')[0].upper()
    if base in STOCK_COMMODITY:
        return True
    if looks_like_non_coin(base):
        return True
    return False

# =====================================================================
def get_top_volume_symbols(top_n=TOP_N):
    try:
        tickers = exchange.fetch_tickers()
    except Exception as e:
        print(f"종목 목록 수집 실패: {e}"); return []
    usdt = [{'symbol': s, 'volume': t['quoteVolume']}
            for s, t in tickers.items()
            if s.endswith(':USDT') and t.get('quoteVolume') is not None
            and not is_non_coin(s)]   # ★주식/원자재 제외
    if not usdt:
        print("USDT 무기한 종목을 찾지 못했습니다."); return []
    df = pd.DataFrame(usdt).sort_values('volume', ascending=False)
    return df['symbol'].head(top_n).tolist()

def calculate_atr(df, period=14):
    high_low = df['high'] - df['low']
    high_close = (df['high'] - df['close'].shift()).abs()
    low_close = (df['low'] - df['close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    eff = min(period, max(1, len(df) - 1))
    atr = tr.rolling(eff).mean().iloc[-1]
    if pd.isna(atr): atr = tr.mean()
    return float(atr)

def atr_pct(df, atr_value):
    price = df['close'].iloc[-1]
    return atr_value / price * 100 if price > 0 else 0.0

def score_sweep(df, lookback=LOOKBACK):
    if len(df) < lookback + 1: return None
    last = df.iloc[-1]; prev = df.iloc[-(lookback + 1):-1]
    prev_high = prev['high'].max(); prev_low = prev['low'].min()
    avg_vol = prev['volume'].mean()
    vol_ratio = last['volume'] / avg_vol if avg_vol > 0 else 0
    rng = (prev_high - prev_low) or 1e-9
    high_pierce = (last['high'] - prev_high) / rng
    low_pierce  = (prev_low - last['low']) / rng
    is_green = last['close'] > last['open']; is_red = last['close'] < last['open']
    short_score = long_score = 0.0
    if high_pierce > -SWEEP_TOLERANCE and last['close'] < prev_high:
        short_score = max(0, high_pierce + SWEEP_TOLERANCE) * 100 + vol_ratio + (1.0 if is_red else 0)
    if low_pierce > -SWEEP_TOLERANCE and last['close'] > prev_low:
        long_score = max(0, low_pierce + SWEEP_TOLERANCE) * 100 + vol_ratio + (1.0 if is_green else 0)
    if long_score >= short_score and long_score > 0:
        direction, score = "LONG", long_score
    elif short_score > 0:
        direction, score = "SHORT", short_score
    else:
        return None
    pierce = low_pierce if direction == "LONG" else high_pierce
    body_ok = is_green if direction == "LONG" else is_red
    if vol_ratio >= 0.8 and pierce > 0 and body_ok:
        grade = "강함 ★★★"
    elif vol_ratio >= 0.5 and pierce > 0:
        grade = "보통 ★★"
    else:
        grade = "약함 ★"
    return {"direction": direction, "score": round(score, 2),
            "grade": grade, "vol_ratio": round(vol_ratio, 2)}

def analyze_symbol(symbol):
    try:
        ohlcv = exchange.fetch_ohlcv(symbol, TIMEFRAME, limit=LIMIT)
        if len(ohlcv) < LOOKBACK + 1: return None
        df = pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume'])
        atr = calculate_atr(df); atrp = atr_pct(df, atr)
        if atrp < ATR_MIN_PCT:   # ATR 변동성 필터
            return None
        res = score_sweep(df)
        if not res: return None
        entry = float(df['close'].iloc[-1])
        if res["direction"] == "LONG":
            sl, tp = entry - 1.5*atr, entry + 3.0*atr
        else:
            sl, tp = entry + 1.5*atr, entry - 3.0*atr
        return {"Symbol": symbol.split(':')[0], "Signal": res["direction"],
                "Grade": res["grade"], "ATR%": round(atrp, 2),
                "Score": res["score"], "VolRatio": res["vol_ratio"],
                "Entry": round(entry, 4), "StopLoss": round(sl, 4),
                "TakeProfit": round(tp, 4)}
    except Exception:
        return None

# =====================================================================
print(f"[{datetime.now():%Y-%m-%d %H:%M}] Bitget 휩소 스캔 (주식/원자재 이중차단 + ATR≥{ATR_MIN_PCT}%)...")
symbols = get_top_volume_symbols(TOP_N)
print(f"  -> 분석 대상: {len(symbols)}개 (주식/원자재/FX 제외됨)\n")

results = []
for i, s in enumerate(symbols):
    r = analyze_symbol(s)
    if r and GRADE_RANK[r["Grade"]] >= MIN_RANK:
        results.append(r)
    time.sleep(REQUEST_PAUSE)
    if (i + 1) % 10 == 0:
        print(f"  진행: {i+1}/{len(symbols)}")

print("\n" + "=" * 70)
if results:
    dfr = pd.DataFrame(results).sort_values("Score", ascending=False).head(SHOW_TOP)
    print(f"휩소 후보: {len(results)}개 중 상위 {len(dfr)}개\n")
    print(dfr.to_string(index=False))
    print(f"\n[등급] 강함★★★: 꼬리+거래량+방향봉 / 보통★★: 꼬리+거래량 / 약함★: 약한 신호")
    print(f"[ATR%] 변동성. {ATR_MIN_PCT}% 미만은 노이즈로 제외됨")
    print("[필터] 주식/원자재/FX 이중차단(블랙리스트+2글자코드)")
    print("※ 처음 보는 심볼은 진입 전 반드시 직접 확인(코인 맞는지, 큰그림 방향, 뉴스).")
    print("※ 휩소는 단타 신호. 큰 그림(일봉)과 방향 일치하는지 확인 필수.")
else:
    print(f"조건 만족 후보 없음. ATR_MIN_PCT 낮추거나 TIMEFRAME='15m'로.")
    print("→ 추세 강한 휩소 자리가 없다는 뜻일 수 있음. 관망도 정답.")
print("=" * 70)

[2026-07-31 16:51] Bitget 휩소 스캔 (주식/원자재 이중차단 + ATR≥0.5%)...
  -> 분석 대상: 50개 (주식/원자재/FX 제외됨)

  진행: 10/50
  진행: 20/50
  진행: 30/50
  진행: 40/50
  진행: 50/50

휩소 후보: 7개 중 상위 7개

   Symbol Signal Grade  ATR%  Score  VolRatio    Entry  StopLoss  TakeProfit
ONDO/USDT   LONG 보통 ★★  1.22  15.89      0.88   0.3968    0.3895      0.4113
NEAR/USDT  SHORT 보통 ★★  1.19   7.34      1.62   1.6900    1.7201      1.6297
 UNI/USDT   LONG 보통 ★★  2.04   5.09      0.60   4.2770    4.1461      4.5389
DEXE/USDT  SHORT 보통 ★★  4.73   4.59      0.90   2.6180    2.8036      2.2469
 MMT/USDT   LONG  약함 ★ 20.18   3.03      0.38   0.2016    0.1406      0.3236
SKHY/USDT   LONG  약함 ★  3.04   2.32      0.42 148.3700  141.6039    161.9021
NBIS/USDT   LONG  약함 ★  3.78   1.87      0.80 188.8700  178.1718    210.2664

[등급] 강함★★★: 꼬리+거래량+방향봉 / 보통★★: 꼬리+거래량 / 약함★: 약한 신호
[ATR%] 변동성. 0.5% 미만은 노이즈로 제외됨
[필터] 주식/원자재/FX 이중차단(블랙리스트+2글자코드)
※ 처음 보는 심볼은 진입 전 반드시 직접 확인(코인 맞는지, 큰그림 방향, 뉴스).
※ 휩소는 단타 신호. 큰 그림(일봉)과 방향 일치하는지 확인 필수.


In [16]:
# =====================================================================
# 횡보장 평균회귀(Mean Reversion) 전략 + 레짐필터 + 정직한 백테스트
# =====================================================================
# 횡보장 전용: 추세추종을 끄고 "박스권 진동"을 노림
#   롱 = RSI(2) 과매도 + BB하단 터치  ->  중심선 복귀 시 익절
#   숏 = RSI(2) 과매수 + BB상단 터치  ->  중심선 복귀 시 익절
#   레짐필터(ADX): 추세장으로 바뀌면 자동으로 신호 OFF
#   손절: 평균회귀 실패(추세 전환) 시 큰 손실 방지
# 목적: 수익 보장이 아니라, 이 전략의 과거 성과를 정직하게 점검
# =====================================================================

!pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값
# ---------------------------
TIMEFRAME    = '4h'
LIMIT        = 1000
BB_PERIOD    = 20
BB_MULT      = 2.0
RSI_PERIOD   = 2       # 초단기 RSI (평균회귀 핵심)
RSI_BUY      = 10      # 이 이하면 과매도 -> 롱 후보
RSI_SELL     = 90      # 이 이상이면 과매수 -> 숏 후보
ADX_PERIOD   = 14
ADX_MAX      = 25      # ADX 이 위면 추세장 -> 신호 끔
STOP_PCT     = 3.0     # 손절 % (평균회귀 실패 대비)
MAX_HOLD     = 20      # 최대 보유 봉 수
REGIME_FILTER = True   # 레짐필터 ON/OFF
SYMBOLS      = ['BTC/USDT:USDT', 'ETH/USDT:USDT', 'SOL/USDT:USDT']

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

# =====================================================================
# 지표
# =====================================================================
def rsi(series, period=RSI_PERIOD):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return (100 - 100/(1+rs)).fillna(50)

def adx(df, period=ADX_PERIOD):
    h, l, c = df['high'], df['low'], df['close']
    up, down = h.diff(), -l.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()
    plus_di = 100*(plus_dm.rolling(period).mean()/atr)
    minus_di = 100*(minus_dm.rolling(period).mean()/atr)
    dx = 100*(plus_di-minus_di).abs()/(plus_di+minus_di).replace(0, np.nan)
    return dx.rolling(period).mean().fillna(0)

def add_indicators(df):
    df = df.copy()
    ma = df['close'].rolling(BB_PERIOD).mean()
    sd = df['close'].rolling(BB_PERIOD).std()
    df['bb_mid'], df['bb_up'], df['bb_low'] = ma, ma+BB_MULT*sd, ma-BB_MULT*sd
    df['rsi'] = rsi(df['close'])
    df['adx'] = adx(df)
    return df

def generate_signals(df):
    df = df.copy()
    df['signal'] = None
    for i in range(len(df)):
        r = df.iloc[i]
        if pd.isna(r['bb_low']) or pd.isna(r['rsi']) or pd.isna(r['adx']):
            continue
        if REGIME_FILTER and r['adx'] > ADX_MAX:   # 추세장이면 평균회귀 끔
            continue
        if r['rsi'] <= RSI_BUY and r['close'] <= r['bb_low']*1.01:
            df.iloc[i, df.columns.get_loc('signal')] = 'LONG'
        elif r['rsi'] >= RSI_SELL and r['close'] >= r['bb_up']*0.99:
            df.iloc[i, df.columns.get_loc('signal')] = 'SHORT'
    return df

# =====================================================================
# 백테스트 (다음봉 진입, 손절+중심선 익절+타임아웃)
# =====================================================================
def backtest(df):
    df = df.reset_index(drop=True)
    trades = []
    i = 0
    while i < len(df)-1:
        sig = df.iloc[i]['signal']
        if sig in ('LONG','SHORT'):
            entry = df.iloc[i+1]['open']
            stop = entry*(1-STOP_PCT/100) if sig=='LONG' else entry*(1+STOP_PCT/100)
            exit_price = result = None
            for j in range(i+1, min(i+1+MAX_HOLD, len(df))):
                mid, c = df.iloc[j]['bb_mid'], df.iloc[j]['close']
                hi, lo = df.iloc[j]['high'], df.iloc[j]['low']
                if sig=='LONG' and lo <= stop:  exit_price, result = stop, 'STOP'; break
                if sig=='SHORT' and hi >= stop: exit_price, result = stop, 'STOP'; break
                if pd.isna(mid): continue
                if sig=='LONG' and c >= mid:  exit_price, result = c, 'EXIT_MID'; break
                if sig=='SHORT' and c <= mid: exit_price, result = c, 'EXIT_MID'; break
            else:
                jl = min(i+MAX_HOLD, len(df)-1)
                exit_price, result, j = df.iloc[jl]['close'], 'TIMEOUT', jl
            pnl = (exit_price-entry)/entry if sig=='LONG' else (entry-exit_price)/entry
            trades.append({'signal': sig, 'result': result, 'pnl_pct': pnl*100, 'win': pnl>0})
            i = j; continue
        i += 1
    return pd.DataFrame(trades)

def summarize(trades):
    if len(trades)==0: return None
    n = len(trades); wins = trades['win'].sum()
    cum = (1+trades['pnl_pct']/100).cumprod()
    mdd = ((cum-cum.cummax())/cum.cummax()).min()*100
    # 손익비 (평균이익 / 평균손실)
    avg_w = trades[trades['win']]['pnl_pct'].mean() if wins else 0
    avg_l = trades[~trades['win']]['pnl_pct'].mean() if (n-wins) else 0
    rr = abs(avg_w/avg_l) if avg_l else 0
    return {"거래수": n, "승률%": round(wins/n*100,1),
            "총수익%": round((cum.iloc[-1]-1)*100,1),
            "손익비": round(rr,2),
            "최대낙폭%": round(mdd,1)}

# =====================================================================
# 실행
# =====================================================================
print(f"[{datetime.now():%Y-%m-%d %H:%M}] 횡보장 평균회귀 백테스트 (레짐필터={'ON' if REGIME_FILTER else 'OFF'})")
print(f"  RSI({RSI_PERIOD}) {RSI_BUY}/{RSI_SELL} | BB({BB_PERIOD},{BB_MULT}) | ADX<{ADX_MAX} | 손절{STOP_PCT}%\n")

summaries = []
for sym in SYMBOLS:
    try:
        ohlcv = exchange.fetch_ohlcv(sym, TIMEFRAME, limit=LIMIT)
        df = add_indicators(pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume']))
        df = generate_signals(df)
        # 현재 시점 신호도 표시
        cur = df.iloc[-1]['signal']
        cur_adx = df.iloc[-1]['adx']
        regime = "추세장(평균회귀 부적합)" if cur_adx > ADX_MAX else "횡보장(평균회귀 적합)"
        trades = backtest(df)
        s = summarize(trades)
        name = sym.split(':')[0]
        if s:
            summaries.append({"종목": name, **s})
        print(f"  {name}: 현재 ADX={cur_adx:.0f} [{regime}] | 현재신호={cur or '없음'}")
        print(f"        백테스트 -> {s}")
    except Exception as e:
        print(f"  {sym} 실패: {e}")
    time.sleep(0.3)

print("\n" + "="*70)
if summaries:
    print(pd.DataFrame(summaries).to_string(index=False))
print("""
[읽는 법 - 평균회귀 전략의 특성]
- 승률은 보통 높게 나옴(자잘하게 자주 이김). 하지만 승률에 속지 마세요.
- ★손익비가 핵심: 1.0 미만이면 "작은 이익 여러 번 < 큰 손실 몇 번" = 결국 손실.
  평균회귀는 이 함정에 빠지기 쉬움. 손익비 > 1 이고 총수익 + 일 때만 유효.
- 현재 ADX로 지금이 횡보장인지 자동 판단. 추세장이면 이 전략 쓰지 마세요.
- REGIME_FILTER, STOP_PCT, RSI_BUY/SELL을 바꿔가며 어떤 조합이 손익비>1 인지 탐색.
""")
print("="*70)
print("※ 과거 성과는 미래를 보장하지 않습니다. 수수료/슬리피지/펀딩비 미반영.")
print("  이건 수익기계가 아니라 전략 점검 도구입니다. 실거래 전 소액 검증 필수.")

[2026-07-31 16:52] 횡보장 평균회귀 백테스트 (레짐필터=ON)
  RSI(2) 10/90 | BB(20,2.0) | ADX<25 | 손절3.0%

  BTC/USDT: 현재 ADX=31 [추세장(평균회귀 부적합)] | 현재신호=없음
        백테스트 -> {'거래수': 7, '승률%': np.float64(57.1), '총수익%': np.float64(-1.1), '손익비': np.float64(0.66), '최대낙폭%': -3.2}
  ETH/USDT: 현재 ADX=22 [횡보장(평균회귀 적합)] | 현재신호=LONG
        백테스트 -> {'거래수': 14, '승률%': np.float64(57.1), '총수익%': np.float64(-0.9), '손익비': np.float64(0.73), '최대낙폭%': -7.0}
  SOL/USDT: 현재 ADX=23 [횡보장(평균회귀 적합)] | 현재신호=없음
        백테스트 -> {'거래수': 7, '승률%': np.float64(28.6), '총수익%': np.float64(-12.0), '손익비': np.float64(0.41), '최대낙폭%': -10.3}

      종목  거래수  승률%  총수익%  손익비  최대낙폭%
BTC/USDT    7 57.1  -1.1 0.66   -3.2
ETH/USDT   14 57.1  -0.9 0.73   -7.0
SOL/USDT    7 28.6 -12.0 0.41  -10.3

[읽는 법 - 평균회귀 전략의 특성]
- 승률은 보통 높게 나옴(자잘하게 자주 이김). 하지만 승률에 속지 마세요.
- ★손익비가 핵심: 1.0 미만이면 "작은 이익 여러 번 < 큰 손실 몇 번" = 결국 손실.
  평균회귀는 이 함정에 빠지기 쉬움. 손익비 > 1 이고 총수익 + 일 때만 유효.
- 현재 ADX로 지금이 횡보장인지 자동 판단. 추세장이면 이 전략 쓰지 마세요.
- REGIME_FILTER, STOP_PCT, RSI_BUY/SELL을 바꿔가며 

In [17]:
# =====================================================================
# 레짐 모니터 (Regime Monitor)  [검증 완료]
# - BTC + 주요 알트의 "지금 상승/하락/횡보"를 한 화면에 자동 판정
# - 방향이 불명확하면 "관망(RANGE)" 신호
# - 매일 차트 안 보고 이것만 돌려서 시장 국면 체크용
# =====================================================================
# 판정 원리: 3가지 지표의 "투표"로 결정 (단일 지표 의존 X)
#   1) ADX     : 추세의 "강도" (방향 아님). 낮으면 횡보
#   2) EMA+DI  : 추세의 "방향" (위/아래)
#   3) OBV     : 자금 흐름 방향 (보조 확인)
#   -> ADX로 추세 유무를 먼저 보고, 방향은 투표로 합의. 애매하면 관망.
# =====================================================================

!pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값
# ---------------------------
TIMEFRAME   = '1d'      # 레짐 판단 주기 ('4h','1d' 권장. 길수록 안정적)
LIMIT       = 300       # 받아올 캔들 수 (200EMA 위해 최소 250+)
ADX_PERIOD  = 14
ADX_TREND   = 25        # 이 이상이면 추세장 후보
EMA_FAST    = 50
EMA_SLOW    = 200
OBV_LOOKBACK = 20
SYMBOLS = ['BTC/USDT:USDT', 'ETH/USDT:USDT', 'SOL/USDT:USDT',
           'XRP/USDT:USDT', 'BNB/USDT:USDT', 'DOGE/USDT:USDT']

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

# =====================================================================
# 지표
# =====================================================================
def ema(s, p): return s.ewm(span=p, adjust=False).mean()

def adx_di(df, period=ADX_PERIOD):
    h, l, c = df['high'], df['low'], df['close']
    up, down = h.diff(), -l.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.ewm(span=period, adjust=False).mean()
    pdi = 100 * plus_dm.ewm(span=period, adjust=False).mean() / atr
    mdi = 100 * minus_dm.ewm(span=period, adjust=False).mean() / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean().fillna(0), pdi.fillna(0), mdi.fillna(0)

def obv_slope(df, lookback=OBV_LOOKBACK):
    d = np.sign(df['close'].diff()).fillna(0)
    obv = (d * df['volume']).cumsum().tail(lookback).values
    if len(obv) < 2: return 0.0
    x = np.arange(len(obv))
    denom = abs(obv[0]) if obv[0] != 0 else 1
    return float(np.polyfit(x, obv/denom, 1)[0])

def judge_regime(df):
    ema_slow_eff = min(EMA_SLOW, len(df)-1)
    adx, pdi, mdi = adx_di(df)
    df = df.copy()
    df['ema_fast'] = ema(df['close'], EMA_FAST)
    df['ema_slow'] = ema(df['close'], ema_slow_eff)
    last = df.iloc[-1]
    cur_adx, cur_pdi, cur_mdi = adx.iloc[-1], pdi.iloc[-1], mdi.iloc[-1]
    obv_s = obv_slope(df)

    bull = bear = 0
    if last['ema_fast'] > last['ema_slow']: bull += 1
    else: bear += 1
    if last['close'] > last['ema_slow']: bull += 1
    else: bear += 1
    if cur_pdi > cur_mdi: bull += 1
    else: bear += 1
    if obv_s > 0.001: bull += 1
    elif obv_s < -0.001: bear += 1

    strong = cur_adx >= ADX_TREND + 5
    border = ADX_TREND <= cur_adx < ADX_TREND + 5

    if cur_adx < ADX_TREND:
        regime = "UPTREND" if bull >= 4 else "DOWNTREND" if bear >= 4 else "RANGE"
    elif strong and bull > bear: regime = "UPTREND"
    elif strong and bear > bull: regime = "DOWNTREND"
    elif border and bull >= 3 and bull > bear: regime = "UPTREND"
    elif border and bear >= 3 and bear > bull: regime = "DOWNTREND"
    else: regime = "RANGE"

    return {"regime": regime, "adx": round(float(cur_adx),1),
            "bull": bull, "bear": bear, "obv_slope": round(obv_s,4),
            "vs_200ema": "위" if last['close'] > last['ema_slow'] else "아래"}

ACTION = {
    "UPTREND":   "상승추세 → 추세추종 유효 / 롱 우위",
    "DOWNTREND": "하락추세 → 관망 또는 숏 / 롱 금물",
    "RANGE":     "횡보·불명확 → 관망 권장 (방향 확정 대기)",
}
ICON = {"UPTREND": "▲ 상승", "DOWNTREND": "▼ 하락", "RANGE": "■ 횡보"}

# =====================================================================
# 실행
# =====================================================================
print(f"[{datetime.now():%Y-%m-%d %H:%M}] 레짐 모니터 ({TIMEFRAME} 기준)\n")
rows = []
for sym in SYMBOLS:
    try:
        ohlcv = exchange.fetch_ohlcv(sym, TIMEFRAME, limit=LIMIT)
        df = pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume'])
        r = judge_regime(df)
        rows.append({"종목": sym.split(':')[0], "레짐": ICON[r['regime']],
                     "ADX": r['adx'], "방향투표": f"{r['bull']}:{r['bear']}",
                     "200EMA": r['vs_200ema'], "OBV기울기": r['obv_slope']})
    except Exception as e:
        print(f"  {sym} 실패: {e}")
    time.sleep(0.3)

print("="*70)
if rows:
    dfr = pd.DataFrame(rows)
    print(dfr.to_string(index=False))
    print("="*70)
    # 시장 전체 종합
    regimes = [r['레짐'] for r in rows]
    up = sum('상승' in x for x in regimes)
    down = sum('하락' in x for x in regimes)
    rng = sum('횡보' in x for x in regimes)
    print(f"\n[시장 종합] 상승 {up}개 / 하락 {down}개 / 횡보 {rng}개")
    if down >= len(rows)*0.6:
        verdict = "시장 전반 하락·약세 → 관망 권장. 신규 롱 자제."
    elif up >= len(rows)*0.6:
        verdict = "시장 전반 상승 → 추세추종 전략 검토 가능 구간."
    else:
        verdict = "방향 혼재·불명확 → 관망 권장. 레짐 확정될 때까지 대기."
    print(f"  >>> {verdict}")
    print("\n[해석] ADX≥30=강한추세 / 25~30=경계 / <25=횡보")
    print("       방향투표 = 상승신호:하락신호 (4:0이면 만장일치)")
    print("       각 종목 행동지침:")
    for r in rows:
        reg = 'UPTREND' if '상승' in r['레짐'] else 'DOWNTREND' if '하락' in r['레짐'] else 'RANGE'
        print(f"         - {r['종목']:5s}: {ACTION[reg]}")
print("="*70)
print("\n※ 추세 판정은 후행적입니다(이미 진행된 추세를 확인). 전환점을")
print("  예측하지 않습니다. 매매 신호가 아니라 '국면 인식' 보조 도구입니다.")

[2026-07-31 16:52] 레짐 모니터 (1d 기준)

       종목   레짐  ADX 방향투표 200EMA  OBV기울기
 BTC/USDT ■ 횡보 13.3  1:3     아래  0.0015
 ETH/USDT ▼ 하락 25.2  1:3     아래  0.0061
 SOL/USDT ▼ 하락 18.4  0:4     아래 -0.0179
 XRP/USDT ■ 횡보 16.8  1:3     아래  0.0183
 BNB/USDT ▼ 하락 34.9  1:2     아래 -0.0007
DOGE/USDT ▼ 하락 26.2  0:4     아래 -0.0045

[시장 종합] 상승 0개 / 하락 4개 / 횡보 2개
  >>> 시장 전반 하락·약세 → 관망 권장. 신규 롱 자제.

[해석] ADX≥30=강한추세 / 25~30=경계 / <25=횡보
       방향투표 = 상승신호:하락신호 (4:0이면 만장일치)
       각 종목 행동지침:
         - BTC/USDT: 횡보·불명확 → 관망 권장 (방향 확정 대기)
         - ETH/USDT: 하락추세 → 관망 또는 숏 / 롱 금물
         - SOL/USDT: 하락추세 → 관망 또는 숏 / 롱 금물
         - XRP/USDT: 횡보·불명확 → 관망 권장 (방향 확정 대기)
         - BNB/USDT: 하락추세 → 관망 또는 숏 / 롱 금물
         - DOGE/USDT: 하락추세 → 관망 또는 숏 / 롱 금물

※ 추세 판정은 후행적입니다(이미 진행된 추세를 확인). 전환점을
  예측하지 않습니다. 매매 신호가 아니라 '국면 인식' 보조 도구입니다.


In [18]:
# =====================================================================
# 레짐 모니터 (Regime Monitor)  [검증 완료]
# - BTC + 주요 알트의 "지금 상승/하락/횡보"를 한 화면에 자동 판정
# - 방향이 불명확하면 "관망(RANGE)" 신호
# - 매일 차트 안 보고 이것만 돌려서 시장 국면 체크용
# =====================================================================
# 판정 원리: 3가지 지표의 "투표"로 결정 (단일 지표 의존 X)
#   1) ADX     : 추세의 "강도" (방향 아님). 낮으면 횡보
#   2) EMA+DI  : 추세의 "방향" (위/아래)
#   3) OBV     : 자금 흐름 방향 (보조 확인)
#   -> ADX로 추세 유무를 먼저 보고, 방향은 투표로 합의. 애매하면 관망.
# =====================================================================

!pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값
# ---------------------------
TIMEFRAME   = '1d'      # 레짐 판단 주기 ('4h','1d' 권장. 길수록 안정적)
LIMIT       = 300       # 받아올 캔들 수 (200EMA 위해 최소 250+)
ADX_PERIOD  = 14
ADX_TREND   = 25        # 이 이상이면 추세장 후보
EMA_FAST    = 50
EMA_SLOW    = 200
OBV_LOOKBACK = 20
SYMBOLS = ['BTC/USDT:USDT', 'ETH/USDT:USDT', 'SOL/USDT:USDT',
           'XRP/USDT:USDT', 'BNB/USDT:USDT', 'DOGE/USDT:USDT']

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

# =====================================================================
# 지표
# =====================================================================
def ema(s, p): return s.ewm(span=p, adjust=False).mean()

def adx_di(df, period=ADX_PERIOD):
    h, l, c = df['high'], df['low'], df['close']
    up, down = h.diff(), -l.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.ewm(span=period, adjust=False).mean()
    pdi = 100 * plus_dm.ewm(span=period, adjust=False).mean() / atr
    mdi = 100 * minus_dm.ewm(span=period, adjust=False).mean() / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean().fillna(0), pdi.fillna(0), mdi.fillna(0)

def obv_slope(df, lookback=OBV_LOOKBACK):
    d = np.sign(df['close'].diff()).fillna(0)
    obv = (d * df['volume']).cumsum().tail(lookback).values
    if len(obv) < 2: return 0.0
    x = np.arange(len(obv))
    denom = abs(obv[0]) if obv[0] != 0 else 1
    return float(np.polyfit(x, obv/denom, 1)[0])

def judge_regime(df):
    ema_slow_eff = min(EMA_SLOW, len(df)-1)
    adx, pdi, mdi = adx_di(df)
    df = df.copy()
    df['ema_fast'] = ema(df['close'], EMA_FAST)
    df['ema_slow'] = ema(df['close'], ema_slow_eff)
    last = df.iloc[-1]
    cur_adx, cur_pdi, cur_mdi = adx.iloc[-1], pdi.iloc[-1], mdi.iloc[-1]
    obv_s = obv_slope(df)

    bull = bear = 0
    if last['ema_fast'] > last['ema_slow']: bull += 1
    else: bear += 1
    if last['close'] > last['ema_slow']: bull += 1
    else: bear += 1
    if cur_pdi > cur_mdi: bull += 1
    else: bear += 1
    if obv_s > 0.001: bull += 1
    elif obv_s < -0.001: bear += 1

    strong = cur_adx >= ADX_TREND + 5
    border = ADX_TREND <= cur_adx < ADX_TREND + 5

    if cur_adx < ADX_TREND:
        regime = "UPTREND" if bull >= 4 else "DOWNTREND" if bear >= 4 else "RANGE"
    elif strong and bull > bear: regime = "UPTREND"
    elif strong and bear > bull: regime = "DOWNTREND"
    elif border and bull >= 3 and bull > bear: regime = "UPTREND"
    elif border and bear >= 3 and bear > bull: regime = "DOWNTREND"
    else: regime = "RANGE"

    return {"regime": regime, "adx": round(float(cur_adx),1),
            "bull": bull, "bear": bear, "obv_slope": round(obv_s,4),
            "vs_200ema": "위" if last['close'] > last['ema_slow'] else "아래"}

ACTION = {
    "UPTREND":   "상승추세 → 추세추종 유효 / 롱 우위",
    "DOWNTREND": "하락추세 → 관망 또는 숏 / 롱 금물",
    "RANGE":     "횡보·불명확 → 관망 권장 (방향 확정 대기)",
}
ICON = {"UPTREND": "▲ 상승", "DOWNTREND": "▼ 하락", "RANGE": "■ 횡보"}

# =====================================================================
# 실행
# =====================================================================
print(f"[{datetime.now():%Y-%m-%d %H:%M}] 레짐 모니터 ({TIMEFRAME} 기준)\n")
rows = []
for sym in SYMBOLS:
    try:
        ohlcv = exchange.fetch_ohlcv(sym, TIMEFRAME, limit=LIMIT)
        df = pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume'])
        r = judge_regime(df)
        rows.append({"종목": sym.split(':')[0], "레짐": ICON[r['regime']],
                     "ADX": r['adx'], "방향투표": f"{r['bull']}:{r['bear']}",
                     "200EMA": r['vs_200ema'], "OBV기울기": r['obv_slope']})
    except Exception as e:
        print(f"  {sym} 실패: {e}")
    time.sleep(0.3)

print("="*70)
if rows:
    dfr = pd.DataFrame(rows)
    print(dfr.to_string(index=False))
    print("="*70)
    # 시장 전체 종합
    regimes = [r['레짐'] for r in rows]
    up = sum('상승' in x for x in regimes)
    down = sum('하락' in x for x in regimes)
    rng = sum('횡보' in x for x in regimes)
    print(f"\n[시장 종합] 상승 {up}개 / 하락 {down}개 / 횡보 {rng}개")
    if down >= len(rows)*0.6:
        verdict = "시장 전반 하락·약세 → 관망 권장. 신규 롱 자제."
    elif up >= len(rows)*0.6:
        verdict = "시장 전반 상승 → 추세추종 전략 검토 가능 구간."
    else:
        verdict = "방향 혼재·불명확 → 관망 권장. 레짐 확정될 때까지 대기."
    print(f"  >>> {verdict}")
    print("\n[해석] ADX≥30=강한추세 / 25~30=경계 / <25=횡보")
    print("       방향투표 = 상승신호:하락신호 (4:0이면 만장일치)")
    print("       각 종목 행동지침:")
    for r in rows:
        reg = 'UPTREND' if '상승' in r['레짐'] else 'DOWNTREND' if '하락' in r['레짐'] else 'RANGE'
        print(f"         - {r['종목']:5s}: {ACTION[reg]}")
print("="*70)
print("\n※ 추세 판정은 후행적입니다(이미 진행된 추세를 확인). 전환점을")
print("  예측하지 않습니다. 매매 신호가 아니라 '국면 인식' 보조 도구입니다.")

[2026-07-31 16:52] 레짐 모니터 (1d 기준)

       종목   레짐  ADX 방향투표 200EMA  OBV기울기
 BTC/USDT ■ 횡보 13.3  1:3     아래  0.0015
 ETH/USDT ▼ 하락 25.2  1:3     아래  0.0061
 SOL/USDT ▼ 하락 18.4  0:4     아래 -0.0179
 XRP/USDT ■ 횡보 16.8  1:3     아래  0.0183
 BNB/USDT ▼ 하락 34.9  1:2     아래 -0.0007
DOGE/USDT ▼ 하락 26.2  0:4     아래 -0.0045

[시장 종합] 상승 0개 / 하락 4개 / 횡보 2개
  >>> 시장 전반 하락·약세 → 관망 권장. 신규 롱 자제.

[해석] ADX≥30=강한추세 / 25~30=경계 / <25=횡보
       방향투표 = 상승신호:하락신호 (4:0이면 만장일치)
       각 종목 행동지침:
         - BTC/USDT: 횡보·불명확 → 관망 권장 (방향 확정 대기)
         - ETH/USDT: 하락추세 → 관망 또는 숏 / 롱 금물
         - SOL/USDT: 하락추세 → 관망 또는 숏 / 롱 금물
         - XRP/USDT: 횡보·불명확 → 관망 권장 (방향 확정 대기)
         - BNB/USDT: 하락추세 → 관망 또는 숏 / 롱 금물
         - DOGE/USDT: 하락추세 → 관망 또는 숏 / 롱 금물

※ 추세 판정은 후행적입니다(이미 진행된 추세를 확인). 전환점을
  예측하지 않습니다. 매매 신호가 아니라 '국면 인식' 보조 도구입니다.


In [19]:
# =====================================================================
# 리스크 우선 숏(Short) 진입 계획기  [검증 완료 / 학습용]
# =====================================================================
# "안전한 숏"은 없습니다. 이건 위험을 '관리'하는 도구입니다.
# 핵심 발상: 얼마 벌지가 아니라 "얼마까지 잃을지"를 먼저 정하고,
#            거기서 역산해 포지션 크기를 계산 (리스크 우선 사이징)
#
# 2단계로 작동:
#   1) 자격 게이트: 진짜 하락추세인지 검증. 아니면 진입 자체를 막음
#   2) 포지션 계획: 계좌의 N%만 잃도록 수량/레버리지/손절/익절 산출
# =====================================================================

!pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값 ★여기를 본인 상황에 맞게★
# ---------------------------
ACCOUNT_SIZE = 1_000_000   # 계좌 전체 금액(원). 학습용이면 실제 넣을 소액으로
RISK_PCT     = 1.0         # 한 거래에서 잃어도 되는 비율(%). 1% 권장, 학습이면 0.5%
MAX_LEVERAGE = 2.0         # 레버리지 상한. 학습용은 1~2배 강력 권장
SL_ATR_MULT  = 1.5         # 손절 = 진입 + 이값*ATR
TP_ATR_MULT  = 3.0         # 익절 = 진입 - 이값*ATR (손익비 2:1)
ADX_MIN      = 20          # 이 미만이면 추세 약함 -> 숏 자격 박탈

TIMEFRAME    = '4h'        # 추세 판단 주기
LIMIT        = 300
SYMBOLS      = ['BTC/USDT:USDT', 'XRP/USDT:USDT', 'DOGE/USDT:USDT',
                'SOL/USDT:USDT', 'BNB/USDT:USDT']  # ETH는 방향 불명확이라 제외

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

# =====================================================================
# 지표
# =====================================================================
def ema(s, p): return s.ewm(span=p, adjust=False).mean()

def adx_di(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    up, down = h.diff(), -l.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    a = tr.ewm(span=period, adjust=False).mean()
    pdi = 100 * plus_dm.ewm(span=period, adjust=False).mean() / a
    mdi = 100 * minus_dm.ewm(span=period, adjust=False).mean() / a
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean().fillna(0), pdi.fillna(0), mdi.fillna(0)

def atr(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

# =====================================================================
# 1) 숏 자격 게이트 (하락추세 확인)
# =====================================================================
def check_short_eligible(df):
    a, pdi, mdi = adx_di(df)
    df = df.copy()
    df['ema50'] = ema(df['close'], 50)
    df['ema200'] = ema(df['close'], min(200, len(df)-1))
    last = df.iloc[-1]
    cur_adx = a.iloc[-1]
    reasons = []; ok = True
    if last['close'] >= last['ema200']:
        ok = False; reasons.append("가격이 200EMA 위")
    if pdi.iloc[-1] >= mdi.iloc[-1]:
        ok = False; reasons.append("+DI>=-DI")
    if last['ema50'] >= last['ema200']:
        ok = False; reasons.append("50EMA>=200EMA(역배열 아님)")
    if cur_adx < ADX_MIN:
        ok = False; reasons.append(f"ADX{cur_adx:.0f}<{ADX_MIN}")
    return ok, round(float(cur_adx),1), reasons

# =====================================================================
# 2) 리스크 우선 포지션 계획
# =====================================================================
def plan_short(entry, atr_val):
    if entry <= 0 or atr_val <= 0:
        return None
    sl = entry + SL_ATR_MULT*atr_val
    tp = entry - TP_ATR_MULT*atr_val
    sl_dist_pct = (sl-entry)/entry
    risk_amt = ACCOUNT_SIZE*(RISK_PCT/100)
    pos_val = risk_amt/sl_dist_pct
    max_pos = ACCOUNT_SIZE*MAX_LEVERAGE
    capped = pos_val > max_pos
    if capped:
        pos_val = max_pos
        risk_amt = pos_val*sl_dist_pct
    return {
        "진입가": round(entry,4),
        "손절가": round(sl,4),
        "익절가": round(tp,4),
        "손절거리%": round(sl_dist_pct*100,2),
        "포지션가치": round(pos_val,0),
        "필요마진": round(pos_val/MAX_LEVERAGE,0),
        "코인수량": round(pos_val/entry,6),
        "실제리스크": round(risk_amt,0),
        "리스크%": round(risk_amt/ACCOUNT_SIZE*100,2),
        "잠재이익": round(pos_val*TP_ATR_MULT*(atr_val/entry),0),
        "손익비": round(TP_ATR_MULT/SL_ATR_MULT,2),
        "레버리지캡": capped,
    }

# =====================================================================
# 실행
# =====================================================================
print(f"[{datetime.now():%Y-%m-%d %H:%M}] 리스크 우선 숏 계획기")
print(f"  계좌 {ACCOUNT_SIZE:,}원 | 거래당 리스크 {RISK_PCT}% | 레버리지 최대 {MAX_LEVERAGE}배\n")

for sym in SYMBOLS:
    try:
        ohlcv = exchange.fetch_ohlcv(sym, TIMEFRAME, limit=LIMIT)
        df = pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume'])
        ok, cur_adx, reasons = check_short_eligible(df)
        name = sym.split(':')[0]
        entry = float(df['close'].iloc[-1])
        atr_val = float(atr(df).iloc[-1])
        print(f"── {name} (ADX {cur_adx}) " + "─"*30)
        if not ok:
            print(f"   ✗ 숏 부적격: {', '.join(reasons)}\n")
            continue
        plan = plan_short(entry, atr_val)
        print(f"   ✓ 하락추세 확인 → 숏 계획")
        for k, v in plan.items():
            if k == "레버리지캡":
                if v: print(f"      ⚠ 레버리지 상한에 걸림(리스크%는 그래도 통제됨)")
            else:
                print(f"      {k}: {v:,}" if isinstance(v,(int,float)) else f"      {k}: {v}")
        print()
    except Exception as e:
        print(f"  {sym} 실패: {e}\n")
    time.sleep(0.3)

print("="*60)
print("""
[이 도구의 핵심 = 포지션 사이징]
- '얼마 잃을지(리스크%)'를 먼저 정함 → 거기서 수량을 역산
- 그래서 손절에 걸려도 손실은 항상 계좌의 RISK_PCT%로 고정
- 진입가/손절가가 아니라 '리스크 금액'이 일정한 게 핵심
- 이걸 지키면 한 번의 실수로 계좌가 날아가지 않음 (생존)

[숏 학습 체크리스트]
1) 자격 통과한 종목만 (✗는 절대 건드리지 않기)
2) 진입 전 가설을 글로: "왜 떨어진다고 보나, 어디서 틀린 걸 인정할까"
3) 손절가 = 코드가 준 값. 무조건 지키기 (숏은 손실 무제한)
4) 펀딩비/숏스퀴즈 직접 관찰 (학습 포인트)
5) 잃어도 멘탈 안 흔들리는 금액인지 마지막 확인
""")
print("="*60)
print("※ 숏에 안전은 없습니다. 이건 위험 관리 도구지 수익 보장이 아닙니다.")
print("  저는 투자자문가가 아니며, 모든 매매는 본인 판단과 책임입니다.")

[2026-07-31 16:53] 리스크 우선 숏 계획기
  계좌 1,000,000원 | 거래당 리스크 1.0% | 레버리지 최대 2.0배

── BTC/USDT (ADX 32.7) ──────────────────────────────
   ✗ 숏 부적격: 50EMA>=200EMA(역배열 아님)

── XRP/USDT (ADX 23.1) ──────────────────────────────
   ✓ 하락추세 확인 → 숏 계획
      진입가: 1.0613
      손절가: 1.0786
      익절가: 1.0267
      손절거리%: 1.63
      포지션가치: 613,076.0
      필요마진: 306,538.0
      코인수량: 577,665.257693
      실제리스크: 10,000.0
      리스크%: 1.0
      잠재이익: 20,000.0
      손익비: 2.0

── DOGE/USDT (ADX 30.1) ──────────────────────────────
   ✓ 하락추세 확인 → 숏 계획
      진입가: 0.0695
      손절가: 0.0709
      익절가: 0.0668
      손절거리%: 1.96
      포지션가치: 509,461.0
      필요마진: 254,731.0
      코인수량: 7,327,215.518894
      실제리스크: 10,000.0
      리스크%: 1.0
      잠재이익: 20,000.0
      손익비: 2.0

── SOL/USDT (ADX 24.6) ──────────────────────────────
   ✓ 하락추세 확인 → 숏 계획
      진입가: 73.178
      손절가: 74.5013
      익절가: 70.5315
      손절거리%: 1.81
      포지션가치: 553,012.0
      필요마진: 276,506.0
      코인수량: 7,557.081065
      실제리스크: 10,000.0
   

In [20]:
# =====================================================================
# 리스크 우선 롱(Long) 진입 계획기  [검증 완료]
# =====================================================================
# 숏 계획기의 거울 버전. 핵심은 동일: "얼마 잃을지"를 먼저 정하고 역산.
#
# 2단계:
#   1) 자격 게이트: 진짜 상승추세인지 검증 (200EMA 위 + 정배열 + ADX)
#                  + OBV로 "자금이 실제로 들어오는지" 보조 확인
#   2) 포지션 계획: 계좌의 N%만 잃도록 수량/레버리지/손절/익절 산출
#
# ※ 지금은 시장이 하락 우세라 "자격 종목 없음"이 정상.
#   상승장으로 돌 때 쓰려고 미리 준비하는 도구.
# =====================================================================

!pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값 ★본인 상황에 맞게★
# ---------------------------
ACCOUNT_SIZE = 1_000_000
RISK_PCT     = 1.0
MAX_LEVERAGE = 2.0
SL_ATR_MULT  = 1.5
TP_ATR_MULT  = 3.0
ADX_MIN      = 20
TIMEFRAME    = '4h'
LIMIT        = 300
SYMBOLS      = ['BTC/USDT:USDT', 'ETH/USDT:USDT', 'SOL/USDT:USDT',
                'XRP/USDT:USDT', 'BNB/USDT:USDT', 'DOGE/USDT:USDT']

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

# =====================================================================
def ema(s, p): return s.ewm(span=p, adjust=False).mean()

def adx_di(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    up, down = h.diff(), -l.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    a = tr.ewm(span=period, adjust=False).mean()
    pdi = 100 * plus_dm.ewm(span=period, adjust=False).mean() / a
    mdi = 100 * minus_dm.ewm(span=period, adjust=False).mean() / a
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    return dx.ewm(span=period, adjust=False).mean().fillna(0), pdi.fillna(0), mdi.fillna(0)

def atr(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

def obv_slope(df, lookback=20):
    d = np.sign(df['close'].diff()).fillna(0)
    obv = (d*df['volume']).cumsum().tail(lookback).values
    if len(obv)<2: return 0.0
    x=np.arange(len(obv)); denom=abs(obv[0]) if obv[0]!=0 else 1
    return float(np.polyfit(x, obv/denom, 1)[0])

# =====================================================================
# 1) 롱 자격 게이트 (상승추세 확인)
# =====================================================================
def check_long_eligible(df):
    a, pdi, mdi = adx_di(df)
    df = df.copy()
    df['ema50'] = ema(df['close'], 50)
    df['ema200'] = ema(df['close'], min(200, len(df)-1))
    last = df.iloc[-1]; cur_adx = a.iloc[-1]; obv_s = obv_slope(df)
    reasons = []; ok = True
    if last['close'] <= last['ema200']:
        ok=False; reasons.append("가격이 200EMA 아래")
    if pdi.iloc[-1] <= mdi.iloc[-1]:
        ok=False; reasons.append("+DI<=-DI")
    if last['ema50'] <= last['ema200']:
        ok=False; reasons.append("50EMA<=200EMA(정배열 아님)")
    if cur_adx < ADX_MIN:
        ok=False; reasons.append(f"ADX{cur_adx:.0f}<{ADX_MIN}")
    obv_warn = ok and obv_s < 0   # 상승자격인데 자금 유출 = 경고
    return ok, round(float(cur_adx),1), reasons, round(obv_s,4), obv_warn

# =====================================================================
# 2) 리스크 우선 롱 계획
# =====================================================================
def plan_long(entry, atr_val):
    if entry<=0 or atr_val<=0: return None
    sl = entry - SL_ATR_MULT*atr_val
    tp = entry + TP_ATR_MULT*atr_val
    sl_dist_pct = (entry-sl)/entry
    risk_amt = ACCOUNT_SIZE*(RISK_PCT/100)
    pos_val = risk_amt/sl_dist_pct
    max_pos = ACCOUNT_SIZE*MAX_LEVERAGE
    capped = pos_val>max_pos
    if capped:
        pos_val=max_pos; risk_amt=pos_val*sl_dist_pct
    return {
        "진입가": round(entry,4), "손절가": round(sl,4), "익절가": round(tp,4),
        "손절거리%": round(sl_dist_pct*100,2),
        "포지션가치": round(pos_val,0), "필요마진": round(pos_val/MAX_LEVERAGE,0),
        "코인수량": round(pos_val/entry,6),
        "실제리스크": round(risk_amt,0), "리스크%": round(risk_amt/ACCOUNT_SIZE*100,2),
        "잠재이익": round(pos_val*TP_ATR_MULT*(atr_val/entry),0),
        "손익비": round(TP_ATR_MULT/SL_ATR_MULT,2), "레버리지캡": capped,
    }

# =====================================================================
# 실행
# =====================================================================
print(f"[{datetime.now():%Y-%m-%d %H:%M}] 리스크 우선 롱 계획기")
print(f"  계좌 {ACCOUNT_SIZE:,}원 | 거래당 리스크 {RISK_PCT}% | 레버리지 최대 {MAX_LEVERAGE}배\n")

eligible_count = 0
for sym in SYMBOLS:
    try:
        ohlcv = exchange.fetch_ohlcv(sym, TIMEFRAME, limit=LIMIT)
        df = pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume'])
        ok, cur_adx, reasons, obv_s, obv_warn = check_long_eligible(df)
        name = sym.split(':')[0]
        entry = float(df['close'].iloc[-1]); atr_val = float(atr(df).iloc[-1])
        print(f"── {name} (ADX {cur_adx}) " + "─"*30)
        if not ok:
            print(f"   ✗ 롱 부적격: {', '.join(reasons)}\n")
            continue
        eligible_count += 1
        plan = plan_long(entry, atr_val)
        print(f"   ✓ 상승추세 확인 → 롱 계획")
        if obv_warn:
            print(f"   ⚠ OBV 경고: 가격은 오르나 자금기울기 {obv_s} (음수) - 상승 동력 약할 수 있음")
        for k, v in plan.items():
            if k=="레버리지캡":
                if v: print(f"      ⚠ 레버리지 상한 도달(리스크%는 통제됨)")
            else:
                print(f"      {k}: {v:,}" if isinstance(v,(int,float)) else f"      {k}: {v}")
        print()
    except Exception as e:
        print(f"  {sym} 실패: {e}\n")
    time.sleep(0.3)

print("="*60)
if eligible_count == 0:
    print("롱 자격 종목 없음 → 지금은 상승추세가 아님. 관망이 정답.")
    print("(이게 정상입니다. 시장이 상승으로 돌 때 다시 돌리세요.)")
print("""
[핵심 = 리스크 우선 사이징]
- 손절에 걸려도 손실은 항상 계좌의 RISK_PCT%로 고정
- 틀려도 안 죽게 만드는 게 목적 (생존 우선)

[롱 학습 체크리스트]
1) ✓ 자격 통과 종목만. ✗는 건드리지 않기
2) OBV 경고(⚠) 뜨면 신중히 - 가격만 오르고 자금은 빠지는 중일 수 있음
3) 진입 전 가설을 글로, 손절가는 무조건 지키기
4) 잃어도 멘탈 안 흔들리는 금액인지 확인
""")
print("="*60)
print("※ 수익 보장이 아니라 위험 관리 도구입니다. 수수료/펀딩비 미반영.")
print("  저는 투자자문가가 아니며, 모든 매매는 본인 판단과 책임입니다.")

[2026-07-31 16:53] 리스크 우선 롱 계획기
  계좌 1,000,000원 | 거래당 리스크 1.0% | 레버리지 최대 2.0배

── BTC/USDT (ADX 32.7) ──────────────────────────────
   ✗ 롱 부적격: 가격이 200EMA 아래, +DI<=-DI

── ETH/USDT (ADX 24.6) ──────────────────────────────
   ✗ 롱 부적격: +DI<=-DI

── SOL/USDT (ADX 24.6) ──────────────────────────────
   ✗ 롱 부적격: 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)

── XRP/USDT (ADX 23.1) ──────────────────────────────
   ✗ 롱 부적격: 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)

── BNB/USDT (ADX 35.7) ──────────────────────────────
   ✓ 상승추세 확인 → 롱 계획
      진입가: 586.72
      손절가: 577.1571
      익절가: 605.8458
      손절거리%: 1.63
      포지션가치: 613,536.0
      필요마진: 306,768.0
      코인수량: 1,045.7054
      실제리스크: 10,000.0
      리스크%: 1.0
      잠재이익: 20,000.0
      손익비: 2.0

── DOGE/USDT (ADX 30.1) ──────────────────────────────
   ✗ 롱 부적격: 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)


[핵심 = 리스크 우선 사이징]
- 손절에 걸려도 손실은 항상 계좌의 RISK_PCT%로 고정
- 틀려도 안 죽게 만드는 게 목적 (생존 우선)

[롱 학습 체크리스트]
1) ✓ 자격 통과 종목만. ✗는 건드리

In [21]:
# =====================================================================
# 리스크 우선 숏(Short) 실시간 스캐너  [백테스트와 동일 로직](short)
# =====================================================================
# short_backtest_1h_majors.py 와 '완전히 같은' 자격 게이트/사이징을 사용한다.
# 차이는 단 하나:
#   - 백테스트: 과거 모든 봉을 훑으며 가상 매매
#   - 스캐너  : '지금 막 닫힌 최신 봉' 하나만 보고 → 지금 진입 자격이 되는지 판정
#
# 즉 백테스트에서 진입했을 바로 그 조건을, 현재 시점에 적용해 '오늘의 후보'를 뽑는다.
#
# ⚠ 이건 '신호 알림'이지 자동매매가 아니다. 주문을 넣지 않는다.
#    출력된 진입가/손절/익절/수량은 '계획'이며, 실제 체결·관리는 본인이 한다.
# =====================================================================

# !pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값  ★백테스트와 동일하게 유지★
# ---------------------------
ACCOUNT_SIZE = 1_000_000
RISK_PCT     = 1.0
MAX_LEVERAGE = 2.0
SL_ATR_MULT  = 1.5
TP_ATR_MULT  = 3.0
ADX_MIN      = 20

TIMEFRAME    = '1h'         # 백테스트와 동일한 1시간봉
LIMIT        = 300          # 지표(EMA200 등) 안정화에 충분한 만큼만

# 백테스트와 '똑같은' 10종목
SYMBOLS = [
    'BTC/USDT:USDT',
    'ETH/USDT:USDT',
    'SOL/USDT:USDT',
    'XRP/USDT:USDT',
    'DOGE/USDT:USDT',
    'ADA/USDT:USDT',
    'AVAX/USDT:USDT',
    'LINK/USDT:USDT',
    'SUI/USDT:USDT',
    'NEAR/USDT:USDT',
]

# 종목당 마진 상한(백테스트와 동일). 동시에 여러 신호가 떠도 한 종목 독식 방지.
MAX_MARGIN_PCT_PER_SYMBOL = 25.0

# 마지막 '닫힌' 봉만 평가할지 여부.
#   True  = 진행 중인 현재 봉을 버리고, 직전에 '확정된' 봉으로 판정 (백테스트와 동일, 권장)
#   False = 아직 안 닫힌 현재 봉으로 판정 (값이 계속 바뀜, 비추천)
USE_CLOSED_BAR = True

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

# =====================================================================
# 지표 (백테스트와 동일)
# =====================================================================
def ema(s, p):
    return s.ewm(span=p, adjust=False).mean()

def adx_di(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    up, down = h.diff(), -l.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    a = tr.ewm(span=period, adjust=False).mean()
    pdi = 100 * plus_dm.ewm(span=period, adjust=False).mean() / a
    mdi = 100 * minus_dm.ewm(span=period, adjust=False).mean() / a
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    adx = dx.ewm(span=period, adjust=False).mean().fillna(0)
    return adx, pdi.fillna(0), mdi.fillna(0)

def atr(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

def prepare(df):
    df = df.copy()
    adx, pdi, mdi = adx_di(df)
    df['adx']   = adx
    df['pdi']   = pdi
    df['mdi']   = mdi
    df['ema50'] = ema(df['close'], 50)
    df['ema200']= ema(df['close'], 200)
    df['atr']   = atr(df)
    return df

# =====================================================================
# 자격 게이트 (백테스트의 eligible_row 와 '완전히 동일')
# =====================================================================
def eligible_row(row):
    reasons = []
    if np.isnan(row['ema200']) or np.isnan(row['atr']):
        return False, ["지표 미완성(데이터 부족)"]
    if row['close'] >= row['ema200']:
        reasons.append("가격이 200EMA 위")
    if row['pdi'] >= row['mdi']:
        reasons.append("+DI>=-DI")
    if row['ema50'] >= row['ema200']:
        reasons.append("50EMA>=200EMA(역배열 아님)")
    if row['adx'] < ADX_MIN:
        reasons.append(f"ADX{row['adx']:.0f}<{ADX_MIN}")
    return (len(reasons) == 0), reasons

# =====================================================================
# 포지션 계획 (백테스트의 사이징과 '완전히 동일')
#   - 잔고(=계좌) 대비 RISK_PCT% 만 잃도록 역산
#   - 레버리지는 상한으로만 작동
#   - 종목당 마진 상한 적용
# =====================================================================
def plan_short(entry, atr_val, equity=ACCOUNT_SIZE):
    if entry <= 0 or atr_val <= 0:
        return None
    sl = entry + SL_ATR_MULT * atr_val
    tp = entry - TP_ATR_MULT * atr_val
    sl_dist_pct = (sl - entry) / entry

    risk_amt = equity * (RISK_PCT / 100)
    pos_val  = risk_amt / sl_dist_pct

    max_pos = equity * MAX_LEVERAGE          # 레버리지 상한
    capped_lev = pos_val > max_pos
    if capped_lev:
        pos_val = max_pos

    margin = pos_val / MAX_LEVERAGE
    capped_sym = False
    if MAX_MARGIN_PCT_PER_SYMBOL is not None:
        cap = equity * (MAX_MARGIN_PCT_PER_SYMBOL / 100)
        if margin > cap:
            margin = cap
            pos_val = margin * MAX_LEVERAGE
            capped_sym = True

    # 상한에 걸리면 실제 리스크는 1%보다 작아짐 → 재계산해서 정확히 표시
    actual_risk = pos_val * sl_dist_pct

    return {
        "진입가": round(entry, 6),
        "손절가": round(sl, 6),
        "익절가": round(tp, 6),
        "손절거리%": round(sl_dist_pct * 100, 2),
        "포지션가치": round(pos_val, 0),
        "필요마진": round(margin, 0),
        "코인수량": round(pos_val / entry, 6),
        "실제리스크": round(actual_risk, 0),
        "리스크%": round(actual_risk / equity * 100, 2),
        "손익비": round(TP_ATR_MULT / SL_ATR_MULT, 2),
        "레버리지상한": capped_lev,
        "종목마진상한": capped_sym,
    }

# =====================================================================
# 실행: 모든 종목을 스캔해서 '지금 진입 자격' 판정
# =====================================================================
def scan():
    print(f"[{datetime.now():%Y-%m-%d %H:%M}] 숏 진입 스캐너 ({TIMEFRAME}봉, {len(SYMBOLS)}종목)")
    print(f"  계좌 {ACCOUNT_SIZE:,}원 | 리스크 {RISK_PCT}%/건 | 레버리지 ≤{MAX_LEVERAGE}배 "
          f"| SL {SL_ATR_MULT}*ATR / TP {TP_ATR_MULT}*ATR ({TP_ATR_MULT/SL_ATR_MULT:.1f}:1)")
    bar_kind = "직전 확정봉" if USE_CLOSED_BAR else "진행 중 현재봉"
    print(f"  판정 기준: {bar_kind} | 종목당 마진상한 {MAX_MARGIN_PCT_PER_SYMBOL}%\n")

    qualified = []   # 자격 통과 종목
    rejected  = []   # 부적격 종목 (이유 포함)

    for sym in SYMBOLS:
        name = sym.split(':')[0]
        try:
            ohlcv = exchange.fetch_ohlcv(sym, TIMEFRAME, limit=LIMIT)
            df = pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume'])
            if len(df) < 210:
                rejected.append((name, None, ["데이터 부족"]))
                continue
            df = prepare(df)

            # 백테스트와 동일하게 '닫힌 봉'으로 판정 → 마지막 행이 진행중이면 그 직전 행 사용
            idx = -2 if USE_CLOSED_BAR else -1
            row = df.iloc[idx]
            ok, reasons = eligible_row(row)
            cur_adx = float(row['adx'])

            if ok:
                entry = float(row['close'])
                atr_v = float(row['atr'])
                plan = plan_short(entry, atr_v)
                qualified.append((name, cur_adx, plan))
            else:
                rejected.append((name, cur_adx, reasons))
        except Exception as e:
            rejected.append((name, None, [f"조회 실패: {e}"]))
        time.sleep(0.3)

    # ---- 출력: 자격 통과 ----
    print("=" * 70)
    if qualified:
        print(f"✓ 숏 진입 자격 통과: {len(qualified)}종목")
        print("=" * 70)
        for name, cur_adx, plan in qualified:
            print(f"── {name}  (ADX {cur_adx:.1f}) " + "─" * 28)
            for k, v in plan.items():
                if k in ("레버리지상한", "종목마진상한"):
                    continue
                if isinstance(v, (int, float)):
                    print(f"      {k}: {v:,}")
                else:
                    print(f"      {k}: {v}")
            flags = []
            if plan["레버리지상한"]: flags.append("레버리지 상한 적용")
            if plan["종목마진상한"]: flags.append("종목 마진상한 적용")
            if flags:
                print(f"      ⚠ {' / '.join(flags)} (리스크%는 그만큼 더 작아짐)")
            print()
    else:
        print("✓ 지금 숏 진입 자격을 통과한 종목이 없습니다.")
        print("=" * 70)
        print("  (하락추세 조건이 안 맞는 것뿐. 신호를 '만들어' 진입하지 말 것)\n")

    # ---- 출력: 부적격 (왜 걸렀는지) ----
    print("─" * 70)
    print(f"✗ 부적격 {len(rejected)}종목 (참고용)")
    for name, cur_adx, reasons in rejected:
        adx_str = f"ADX {cur_adx:.1f}" if cur_adx is not None else "ADX -"
        print(f"   {name:6s} ({adx_str}): {', '.join(reasons)}")
    print()

    print("=" * 70)
    print("""[사용 원칙]
1) 자격 통과 종목만 후보다. 부적격은 절대 건드리지 않는다.
2) 출력된 손절가는 '무조건' 지킨다. 숏은 손실이 무제한이다.
3) 동시에 여러 종목이 떠도, 한 계좌 현금 한도 안에서만 나눠 들어간다.
4) 진입 전 한 줄이라도 적는다: "왜 떨어진다고 보나 / 어디서 틀린 걸 인정할까"
5) 이건 신호일 뿐, 자동주문이 아니다. 체결·관리는 본인 몫이다.""")
    print("=" * 70)
    print("※ 백테스트가 좋았다고 미래가 보장되지 않습니다. 최근 한 구간의 결과일 뿐입니다.")
    print("  저는 투자자문가가 아니며, 모든 매매는 본인 판단과 책임입니다.")

if __name__ == '__main__':
    scan()

[2026-07-31 16:53] 숏 진입 스캐너 (1h봉, 10종목)
  계좌 1,000,000원 | 리스크 1.0%/건 | 레버리지 ≤2.0배 | SL 1.5*ATR / TP 3.0*ATR (2.0:1)
  판정 기준: 직전 확정봉 | 종목당 마진상한 25.0%

✓ 숏 진입 자격 통과: 7종목
── BTC/USDT  (ADX 44.0) ────────────────────────────
      진입가: 62,701.3
      손절가: 63,270.621632
      익절가: 61,562.656736
      손절거리%: 0.91
      포지션가치: 500,000.0
      필요마진: 250,000.0
      코인수량: 7.974316
      실제리스크: 4,540.0
      리스크%: 0.45
      손익비: 2.0
      ⚠ 종목 마진상한 적용 (리스크%는 그만큼 더 작아짐)

── ETH/USDT  (ADX 52.8) ────────────────────────────
      진입가: 1,862.1
      손절가: 1,883.095082
      익절가: 1,820.109835
      손절거리%: 1.13
      포지션가치: 500,000.0
      필요마진: 250,000.0
      코인수량: 268.514043
      실제리스크: 5,637.0
      리스크%: 0.56
      손익비: 2.0
      ⚠ 종목 마진상한 적용 (리스크%는 그만큼 더 작아짐)

── SOL/USDT  (ADX 32.1) ────────────────────────────
      진입가: 73.061
      손절가: 73.783005
      익절가: 71.616989
      손절거리%: 0.99
      포지션가치: 500,000.0
      필요마진: 250,000.0
      코인수량: 6,843.596447
      실제리스크: 4,941.0
      리스크%: 0.4

In [22]:
# =====================================================================
# 리스크 우선 롱(Long) 실시간 스캐너  [백테스트와 동일 로직]
# =====================================================================
# long_backtest_2025H2.py 와 '완전히 같은' 자격 게이트/사이징을 사용한다(방향만 롱).
# 차이는 단 하나:
#   - 백테스트: 과거 모든 봉을 훑으며 가상 매매
#   - 스캐너  : '지금 막 닫힌 최신 봉' 하나만 보고 → 지금 진입 자격이 되는지 판정
#
# 즉 백테스트에서 진입했을 바로 그 조건을, 현재 시점에 적용해 '오늘의 후보'를 뽑는다.
#
# ⚠ 롱 전략은 아직 2025-06~10 한 구간에서만 검증됐다. 그 결과를 먼저 확인하고 쓸 것.
# ⚠ 이건 '신호 알림'이지 자동매매가 아니다. 주문을 넣지 않는다.
#    출력된 진입가/손절/익절/수량은 '계획'이며, 실제 체결·관리는 본인이 한다.
# =====================================================================

# !pip install ccxt pandas numpy -q

import ccxt
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ---------------------------
# 설정값  ★백테스트와 동일하게 유지★
# ---------------------------
ACCOUNT_SIZE = 1_000_000
RISK_PCT     = 1.0
MAX_LEVERAGE = 2.0
SL_ATR_MULT  = 1.5
TP_ATR_MULT  = 3.0
ADX_MIN      = 20

TIMEFRAME    = '1h'         # 백테스트와 동일한 1시간봉
LIMIT        = 300          # 지표(EMA200 등) 안정화에 충분한 만큼만

# 백테스트와 '똑같은' 10종목
SYMBOLS = [
    'BTC/USDT:USDT',
    'ETH/USDT:USDT',
    'SOL/USDT:USDT',
    'XRP/USDT:USDT',
    'DOGE/USDT:USDT',
    'ADA/USDT:USDT',
    'AVAX/USDT:USDT',
    'LINK/USDT:USDT',
    'SUI/USDT:USDT',
    'NEAR/USDT:USDT',
]

# 종목당 마진 상한(백테스트와 동일). 동시에 여러 신호가 떠도 한 종목 독식 방지.
MAX_MARGIN_PCT_PER_SYMBOL = 25.0

# 마지막 '닫힌' 봉만 평가할지 여부.
#   True  = 진행 중인 현재 봉을 버리고, 직전에 '확정된' 봉으로 판정 (백테스트와 동일, 권장)
#   False = 아직 안 닫힌 현재 봉으로 판정 (값이 계속 바뀜, 비추천)
USE_CLOSED_BAR = True

exchange = ccxt.bitget({'options': {'defaultType': 'swap'}, 'enableRateLimit': True})

# =====================================================================
# 지표 (백테스트와 동일)
# =====================================================================
def ema(s, p):
    return s.ewm(span=p, adjust=False).mean()

def adx_di(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    up, down = h.diff(), -l.diff()
    plus_dm = ((up > down) & (up > 0)) * up
    minus_dm = ((down > up) & (down > 0)) * down
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    a = tr.ewm(span=period, adjust=False).mean()
    pdi = 100 * plus_dm.ewm(span=period, adjust=False).mean() / a
    mdi = 100 * minus_dm.ewm(span=period, adjust=False).mean() / a
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan)
    adx = dx.ewm(span=period, adjust=False).mean().fillna(0)
    return adx, pdi.fillna(0), mdi.fillna(0)

def atr(df, period=14):
    h, l, c = df['high'], df['low'], df['close']
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

def prepare(df):
    df = df.copy()
    adx, pdi, mdi = adx_di(df)
    df['adx']   = adx
    df['pdi']   = pdi
    df['mdi']   = mdi
    df['ema50'] = ema(df['close'], 50)
    df['ema200']= ema(df['close'], 200)
    df['atr']   = atr(df)
    return df

# =====================================================================
# 자격 게이트 (백테스트의 eligible_row 와 '완전히 동일')
# =====================================================================
def eligible_row(row):
    # 롱 = 상승추세 확인 (숏의 모든 조건을 뒤집음)
    reasons = []
    if np.isnan(row['ema200']) or np.isnan(row['atr']):
        return False, ["지표 미완성(데이터 부족)"]
    if row['close'] <= row['ema200']:
        reasons.append("가격이 200EMA 아래")
    if row['pdi'] <= row['mdi']:
        reasons.append("+DI<=-DI")
    if row['ema50'] <= row['ema200']:
        reasons.append("50EMA<=200EMA(정배열 아님)")
    if row['adx'] < ADX_MIN:
        reasons.append(f"ADX{row['adx']:.0f}<{ADX_MIN}")
    return (len(reasons) == 0), reasons

# =====================================================================
# 포지션 계획 (백테스트의 사이징과 '완전히 동일')
#   - 잔고(=계좌) 대비 RISK_PCT% 만 잃도록 역산
#   - 레버리지는 상한으로만 작동
#   - 종목당 마진 상한 적용
# =====================================================================
def plan_short(entry, atr_val, equity=ACCOUNT_SIZE):
    if entry <= 0 or atr_val <= 0:
        return None
    sl = entry - SL_ATR_MULT * atr_val   # 롱: 손절은 진입 '아래'
    tp = entry + TP_ATR_MULT * atr_val   # 롱: 익절은 진입 '위'
    sl_dist_pct = (entry - sl) / entry   # 손절까지 거리(양수)

    risk_amt = equity * (RISK_PCT / 100)
    pos_val  = risk_amt / sl_dist_pct

    max_pos = equity * MAX_LEVERAGE          # 레버리지 상한
    capped_lev = pos_val > max_pos
    if capped_lev:
        pos_val = max_pos

    margin = pos_val / MAX_LEVERAGE
    capped_sym = False
    if MAX_MARGIN_PCT_PER_SYMBOL is not None:
        cap = equity * (MAX_MARGIN_PCT_PER_SYMBOL / 100)
        if margin > cap:
            margin = cap
            pos_val = margin * MAX_LEVERAGE
            capped_sym = True

    # 상한에 걸리면 실제 리스크는 1%보다 작아짐 → 재계산해서 정확히 표시
    actual_risk = pos_val * sl_dist_pct

    return {
        "진입가": round(entry, 6),
        "손절가": round(sl, 6),
        "익절가": round(tp, 6),
        "손절거리%": round(sl_dist_pct * 100, 2),
        "포지션가치": round(pos_val, 0),
        "필요마진": round(margin, 0),
        "코인수량": round(pos_val / entry, 6),
        "실제리스크": round(actual_risk, 0),
        "리스크%": round(actual_risk / equity * 100, 2),
        "손익비": round(TP_ATR_MULT / SL_ATR_MULT, 2),
        "레버리지상한": capped_lev,
        "종목마진상한": capped_sym,
    }

# =====================================================================
# 실행: 모든 종목을 스캔해서 '지금 진입 자격' 판정
# =====================================================================
def scan():
    print(f"[{datetime.now():%Y-%m-%d %H:%M}] 롱 진입 스캐너 ({TIMEFRAME}봉, {len(SYMBOLS)}종목)")
    print(f"  계좌 {ACCOUNT_SIZE:,}원 | 리스크 {RISK_PCT}%/건 | 레버리지 ≤{MAX_LEVERAGE}배 "
          f"| SL {SL_ATR_MULT}*ATR / TP {TP_ATR_MULT}*ATR ({TP_ATR_MULT/SL_ATR_MULT:.1f}:1)")
    bar_kind = "직전 확정봉" if USE_CLOSED_BAR else "진행 중 현재봉"
    print(f"  판정 기준: {bar_kind} | 종목당 마진상한 {MAX_MARGIN_PCT_PER_SYMBOL}%\n")

    qualified = []   # 자격 통과 종목
    rejected  = []   # 부적격 종목 (이유 포함)

    for sym in SYMBOLS:
        name = sym.split(':')[0]
        try:
            ohlcv = exchange.fetch_ohlcv(sym, TIMEFRAME, limit=LIMIT)
            df = pd.DataFrame(ohlcv, columns=['time','open','high','low','close','volume'])
            if len(df) < 210:
                rejected.append((name, None, ["데이터 부족"]))
                continue
            df = prepare(df)

            # 백테스트와 동일하게 '닫힌 봉'으로 판정 → 마지막 행이 진행중이면 그 직전 행 사용
            idx = -2 if USE_CLOSED_BAR else -1
            row = df.iloc[idx]
            ok, reasons = eligible_row(row)
            cur_adx = float(row['adx'])

            if ok:
                entry = float(row['close'])
                atr_v = float(row['atr'])
                plan = plan_short(entry, atr_v)
                qualified.append((name, cur_adx, plan))
            else:
                rejected.append((name, cur_adx, reasons))
        except Exception as e:
            rejected.append((name, None, [f"조회 실패: {e}"]))
        time.sleep(0.3)

    # ---- 출력: 자격 통과 ----
    print("=" * 70)
    if qualified:
        print(f"✓ 롱 진입 자격 통과: {len(qualified)}종목")
        print("=" * 70)
        for name, cur_adx, plan in qualified:
            print(f"── {name}  (ADX {cur_adx:.1f}) " + "─" * 28)
            for k, v in plan.items():
                if k in ("레버리지상한", "종목마진상한"):
                    continue
                if isinstance(v, (int, float)):
                    print(f"      {k}: {v:,}")
                else:
                    print(f"      {k}: {v}")
            flags = []
            if plan["레버리지상한"]: flags.append("레버리지 상한 적용")
            if plan["종목마진상한"]: flags.append("종목 마진상한 적용")
            if flags:
                print(f"      ⚠ {' / '.join(flags)} (리스크%는 그만큼 더 작아짐)")
            print()
    else:
        print("✓ 지금 롱 진입 자격을 통과한 종목이 없습니다.")
        print("=" * 70)
        print("  (하락추세 조건이 안 맞는 것뿐. 신호를 '만들어' 진입하지 말 것)\n")

    # ---- 출력: 부적격 (왜 걸렀는지) ----
    print("─" * 70)
    print(f"✗ 부적격 {len(rejected)}종목 (참고용)")
    for name, cur_adx, reasons in rejected:
        adx_str = f"ADX {cur_adx:.1f}" if cur_adx is not None else "ADX -"
        print(f"   {name:6s} ({adx_str}): {', '.join(reasons)}")
    print()

    print("=" * 70)
    print("""[사용 원칙]
1) 자격 통과 종목만 후보다. 부적격은 절대 건드리지 않는다.
2) 출력된 손절가는 '무조건' 지킨다. 손절 없는 롱은 큰 하락에 계좌가 녹는다.
3) 동시에 여러 종목이 떠도, 한 계좌 현금 한도 안에서만 나눠 들어간다.
4) 진입 전 한 줄이라도 적는다: "왜 떨어진다고 보나 / 어디서 틀린 걸 인정할까"
5) 이건 신호일 뿐, 자동주문이 아니다. 체결·관리는 본인 몫이다.""")
    print("=" * 70)
    print("※ 백테스트가 좋았다고 미래가 보장되지 않습니다. 최근 한 구간의 결과일 뿐입니다.")
    print("  저는 투자자문가가 아니며, 모든 매매는 본인 판단과 책임입니다.")

if __name__ == '__main__':
    scan()

[2026-07-31 16:53] 롱 진입 스캐너 (1h봉, 10종목)
  계좌 1,000,000원 | 리스크 1.0%/건 | 레버리지 ≤2.0배 | SL 1.5*ATR / TP 3.0*ATR (2.0:1)
  판정 기준: 직전 확정봉 | 종목당 마진상한 25.0%

✓ 지금 롱 진입 자격을 통과한 종목이 없습니다.
  (하락추세 조건이 안 맞는 것뿐. 신호를 '만들어' 진입하지 말 것)

──────────────────────────────────────────────────────────────────────
✗ 부적격 10종목 (참고용)
   BTC/USDT (ADX 44.0): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)
   ETH/USDT (ADX 52.8): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)
   SOL/USDT (ADX 32.1): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)
   XRP/USDT (ADX 48.7): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)
   DOGE/USDT (ADX 45.8): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)
   ADA/USDT (ADX 16.6): +DI<=-DI, ADX17<20
   AVAX/USDT (ADX 15.0): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님), ADX15<20
   LINK/USDT (ADX 56.1): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)
   SUI/USDT (ADX 50.0): 가격이 200EMA 아래, +DI<=-DI, 50EMA<=200EMA(정배열 아님)
   NEAR/USDT (ADX 31.0): 가격이 200EMA 아래, 50EMA<=200EMA(정배열 아님)

[사용 원